# Dataset Forge Critic — private GPU control panel
Run All is safe: it only loads configuration and creates controls. It never starts training. Keep Jupyter and TensorBoard bound to `127.0.0.1` and reach them through authenticated SSH forwarding. Stopping training does **not** stop cloud billing.


In [ ]:
from pathlib import Path
import json, subprocess, sys
from IPython.display import display, JSON
from ipywidgets import Button, Dropdown, Text, VBox, Output
from dataset_forge_critic.cli import _configuration
from dataset_forge_critic.trainer import run_directory
ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
profiles = {p.stem: p for p in sorted((ROOT / 'configs' / 'profiles').glob('*.yaml')) if p.stem != 'cpu-synthetic'}
profile = Dropdown(options=list(profiles), description='Profile')
display(profile)


In [ ]:
summary = Output()
def show_summary(*_):
    summary.clear_output()
    cfg = _configuration(profiles[profile.value])
    with summary:
        display(JSON({'profile': cfg.profile, 'config_digest': cfg.digest(), 'model': cfg.model.model_id, 'revision': cfg.model.revision, 'sequence_length': cfg.max_sequence_length, 'mixture': cfg.mixture.model_dump(), 'limits': cfg.limits.model_dump(), 'estimated_cost_usd': cfg.limits.estimated_process_cost_usd, 'run_dir': str(run_directory(cfg)), 'readiness': 'GPU certification pending'}))
profile.observe(show_summary, names='value'); show_summary(); display(summary)


In [ ]:
confirmation = Text(description='Digest', placeholder='Paste the full displayed config digest for this run')
resume_path = Text(description='Resume', placeholder='Optional checkpoint path')
start = Button(description='Start Training', button_style='danger')
refresh = Button(description='Refresh Status')
stop = Button(description='Graceful Stop', button_style='warning')
controls_output = Output()
def command(action):
    cfg = _configuration(profiles[profile.value]); args = [sys.executable, '-m', 'dataset_forge_critic.cli', action]
    if action == 'launch':
        if confirmation.value != cfg.digest(): raise ValueError('Per-run confirmation must equal the displayed digest')
        args += ['--config', str(profiles[profile.value]), '--confirm', confirmation.value]
        if resume_path.value: args += ['--resume', resume_path.value]
    else: args += ['--run-dir', str(run_directory(cfg))]
    return subprocess.run(args, cwd=ROOT, text=True, capture_output=True)
def handle(action):
    def inner(_):
        controls_output.clear_output();
        with controls_output:
            try:
                result = command(action); print(result.stdout or result.stderr)
            except Exception as exc: print(f'BLOCKED: {exc}')
    return inner
start.on_click(handle('launch')); refresh.on_click(handle('status')); stop.on_click(handle('stop'))
display(VBox([confirmation, resume_path, start, refresh, stop, controls_output]))


## Monitoring
Start TensorBoard separately with `tensorboard --logdir outputs --host 127.0.0.1 --port 6006`. Forward port 6006 over SSH. The status control reports loss, learning rate, steps, processed and supervised token throughput, current/peak VRAM, elapsed time, sampler exposure, exclusions, checkpoints, and stop state. ETA and dollar cost remain estimates derived only after the representative benchmark. Training is a detached process and continues if this browser disconnects.
